# Importing Graph and creating Laplacain Matrix

In [ ]:
import numpy as np
import math

file_path = "email-Enron.txt"
# Load edges from file (assuming whitespace or comma separated: nodeA nodeB per line)
edges = np.loadtxt(file_path, dtype=int)

# Find the number of nodes (assuming nodes are labeled from 0 to N-1, adjust if necessary)
n_nodes = edges.max() + 1
print(f"n_nodes:{n_nodes}")

# Calculate the number of edges
n_edges = len(edges)
print(f"n_edges:{n_edges}")

# Initialize adjacency matrix
adj = np.zeros((n_nodes, n_nodes), dtype=int)

# Fill adjacency matrix (assuming undirected and unweighted)
for a, b in edges:
    adj[a, b] = 1
    adj[b, a] = 1

In [ ]:
# Degree matrix: diagonal with node degrees
degrees = adj.sum(axis=1)
deg_matrix = np.diag(degrees)

# Laplacian: L = D - A
laplacian = deg_matrix - adj

# Calculating all pairs effective resistance matrix

In [ ]:
def effective_resistance_all_pairs(L):
    # Compute Moore-Penrose pseudoinverse of Laplacian
    L_pinv = np.linalg.pinv(L)

    # Diagonal elements of pseudoinverse
    diag = np.diag(L_pinv)
    # print(diag)

    # Compute effective resistance matrix R
    # R[i,j] = L^+_ii + L^+_jj - 2 * L^+_ij
    R = diag[:, np.newaxis] + diag[np.newaxis, :] - 2 * L_pinv
    # print(diag[np.newaxis,:])
    return R

In [ ]:
R = effective_resistance_all_pairs(laplacian)
kirchoff_at_start = R.sum()/(2*math.comb(n_nodes,2))
print(kirchoff_at_start)

## Edge augmentaion by proposed k-median with penalties algorihtm




In [ ]:
k_values_to_test = [5,10, 15,20, 25, 30, 40, 50]
# k_values_to_test = [50]
num_trials_per_k = 10 #number of random source vertcies choosen per each k

print(f"k_values_to_test: {k_values_to_test}")
print(f"num_trials_per_k: {num_trials_per_k}")

### Edge Augmentation via k-medians with penalties algorithm

In [ ]:
import numpy as np
import math

def effective_resistance_all_pairs(L):
    # Compute Moore-Penrose pseudoinverse of Laplacian
    L_pinv = np.linalg.pinv(L)

    # Diagonal elements of pseudoinverse
    diag = np.diag(L_pinv)

    # Compute effective resistance matrix R
    # R[i,j] = L^+_ii + L^+_jj - 2 * L^+_ij
    R = diag[:, np.newaxis] + diag[np.newaxis, :] - 2 * L_pinv
    return R

def _total_cost_penalized(dist_matrix, centers_set, penalties):
    n = dist_matrix.shape[0]
    if not centers_set:
        return np.sum(penalties)

    centers_list = list(centers_set)
    dist_to_centers = dist_matrix[:, centers_list]
    min_distances_to_centers = np.min(dist_to_centers, axis=1);
    costs = np.minimum(min_distances_to_centers, penalties)
    return np.sum(costs)

def _forward_greedy_centers_penalized(dist_matrix, k, penalties):
    n = dist_matrix.shape[0]
    centers = set()
    current_min_distances_or_penalties = penalties.copy()

    for _ in range(k):
        best_candidate = None
        best_total_cost = np.inf

        for candidate in range(n):
            if candidate in centers:
                continue
            temp_centers = centers.union({candidate})
            cost_with_candidate = _total_cost_penalized(dist_matrix, temp_centers, penalties)

            if cost_with_candidate < best_total_cost:
                best_total_cost = cost_with_candidate
                best_candidate = candidate

        if best_candidate is not None:
            centers.add(best_candidate)

    return centers

def k_median_one_swap_until_optimal_penalized(dist_matrix, k, penalties):
    n = dist_matrix.shape[0]
    centers = _forward_greedy_centers_penalized(dist_matrix, k, penalties)
    current_cost = _total_cost_penalized(dist_matrix, centers, penalties)

    while True:
        improved = False
        for center_to_remove in list(centers):
            for candidate_to_add in range(n):
                if candidate_to_add not in centers:
                    new_centers = centers.copy()
                    new_centers.remove(center_to_remove)
                    new_centers.add(candidate_to_add)

                    new_cost = _total_cost_penalized(dist_matrix, new_centers, penalties)

                    if new_cost < current_cost:
                        centers = new_centers
                        current_cost = new_cost
                        improved = True
                        break
            if improved:
                break
        if not improved:
            break

    return list(centers)


all_k_results = {}

for k in k_values_to_test:
    print(f"\n--- Running experiments for k = {k} ---")
    trial_results = []

    for trial in range(num_trials_per_k):
        # 1. Select a random source vertex
        random_source_vertex = np.random.randint(0, n_nodes)

        # 2. Calculate penalties using effective resistance from this source
        penalties = R[random_source_vertex, :]

        # Include the below 4 lines if you wanted to run on the new metric defined for the algorithm,
        # It is just that the without this you will have extra additive factor of approximation of algorithm with less mutlilpicative factor.

        # for i in range(n_nodes):
        #   for j in range(n_nodes):
        #     if i != j  :
        #       R[i,j] = R[i,j] + 1


        # 3. Run the penalized k-median algorithm to find centers
        penalized_centers = k_median_one_swap_until_optimal_penalized(R, k, penalties)

        # 4. Augment the graph by adding edges from the random source vertex to these penalized centers
        adj_augmented = adj.copy()
        actual_new_edges_added = 0
        for center in penalized_centers:
            if adj_augmented[random_source_vertex, center] == 0:
                adj_augmented[random_source_vertex, center] = 1
                adj_augmented[center, random_source_vertex] = 1
                actual_new_edges_added += 1

        # 5. Calculate the augmented Laplacian
        degrees_augmented = adj_augmented.sum(axis=1)
        deg_matrix_augmented = np.diag(degrees_augmented)
        laplacian_augmented = deg_matrix_augmented - adj_augmented

        # 6. Recalculate the effective resistance matrix
        R_augmented = effective_resistance_all_pairs(laplacian_augmented)

        # 7. Calculate the Kirchhoff index for the augmented graph
        kirchoff_augmented = R_augmented.sum()/(2*math.comb(n_nodes,2))

        trial_results.append({
            'trial_id': trial + 1,
            'source_vertex': random_source_vertex,
            'penalized_centers': sorted(penalized_centers),
            'actual_new_edges_added': actual_new_edges_added,
            'kirchoff_index': kirchoff_augmented
        })

        print(f"  Trial {trial+1}: Source {random_source_vertex}, Actual New Edges {actual_new_edges_added}, Kirchhoff Index {kirchoff_augmented:.4f}")

    # After all trials for a given k, identify and store the source vertex that yielded the minimum Kirchhoff index
    min_kirchoff_trial = min(trial_results, key=lambda x: x['kirchoff_index'])

    all_k_results[k] = {
        'min_kirchoff_source_vertex': min_kirchoff_trial['source_vertex'],
        'min_kirchoff_penalized_centers': min_kirchoff_trial['penalized_centers'],
        'min_kirchoff_value': min_kirchoff_trial['kirchoff_index'],
        'actual_new_edges_added_for_min_kirchoff': min_kirchoff_trial['actual_new_edges_added'],
        'all_trials': trial_results # Optionally store all trial results
    }
    print(f"  Minimum Kirchhoff index for k={k}: {min_kirchoff_trial['kirchoff_index']:.4f} (Source: {min_kirchoff_trial['source_vertex']}, New Edges: {min_kirchoff_trial['actual_new_edges_added']})")

print("\n--- All k-value experiments completed ---")
for k, data in all_k_results.items():
    print(f"\nSummary for k={k}:")
    print(f"  Source for Min Kirchhoff: {data['min_kirchoff_source_vertex']}")
    print(f"  Min Kirchhoff Index: {data['min_kirchoff_value']:.4f}")
    print(f"  Actual New Edges Added for Min Kirchhoff: {data['actual_new_edges_added_for_min_kirchoff']}")
    # print(f"  Penalized Centers (Min Kirchhoff): {data['min_kirchoff_penalized_centers']}")